# Imports

In [44]:
from sympy import *
import numpy as np
from eom import EOM
from scipy.linalg import solve_continuous_are

# Define LQR function

In [50]:
def lqr(A: np.ndarray, B: np.ndarray, Q: np.ndarray, R: np.ndarray) -> np.ndarray:
    """Compute the LQR gain matrix K.

    Args:
        A (np.ndarray): State matrix.
        B (np.ndarray): Input matrix.
        Q (np.ndarray): State cost matrix.
        R (np.ndarray): Input cost matrix.

    Returns:
        np.ndarray: LQR gain matrix K.
    """
    P = solve_continuous_are(A, B, Q, R)
    K = np.linalg.inv(R) @ B.T @ P
    return K

# Derive equations of motion symbolically

In [46]:
# Derive EOMs
eoms = EOM()
eoms.derive_eoms_symbolic()
f_sym : Matrix = eoms.f_sym
f_sym

Matrix([
[v_x*cos(psi)*cos(theta) + v_y*(sin(phi)*sin(theta)*cos(psi) - sin(psi)*cos(phi)) + v_z*(sin(phi)*sin(psi) + sin(theta)*cos(phi)*cos(psi))],
[v_x*sin(psi)*cos(theta) + v_y*(sin(phi)*sin(psi)*sin(theta) + cos(phi)*cos(psi)) - v_z*(sin(phi)*cos(psi) - sin(psi)*sin(theta)*cos(phi))],
[                                                                      -v_x*sin(theta) + v_y*sin(phi)*cos(theta) + v_z*cos(phi)*cos(theta)],
[                                                                                                 (w_y*sin(phi) + w_z*cos(phi))/cos(theta)],
[                                                                                                              w_y*cos(phi) - w_z*sin(phi)],
[                                                                                  w_x + w_y*sin(phi)*tan(theta) + w_z*cos(phi)*tan(theta)],
[                                                                                                         g*sin(theta) + v_y*w_z - v_z*w_y],
[   

# Evaluate EoMs numerically at given parameters

In [ ]:
# Params
mass = 1.625  # kg
inertia = np.diag([0.02, 0.02, 0.03])  # kg*m^2
leg_length = 0.15  # m
k_f_val = 62.8 # Slope of thrust/motor torque curve (linear)
k_yaw_val = 1.0 # Yaw torque constant, default 1.0

eoms.set_parameters(mass=mass, inertia=inertia, leg_length=leg_length, k_f_val=k_f_val, k_yaw_val=k_yaw_val)
eoms.derive_eoms_numeric()
f_num : Matrix = eoms.f_num
f_num

Matrix([
[           v_x*cos(psi)*cos(theta) + v_y*(sin(phi)*sin(theta)*cos(psi) - sin(psi)*cos(phi)) + v_z*(sin(phi)*sin(psi) + sin(theta)*cos(phi)*cos(psi))],
[           v_x*sin(psi)*cos(theta) + v_y*(sin(phi)*sin(psi)*sin(theta) + cos(phi)*cos(psi)) - v_z*(sin(phi)*cos(psi) - sin(psi)*sin(theta)*cos(phi))],
[                                                                                 -v_x*sin(theta) + v_y*sin(phi)*cos(theta) + v_z*cos(phi)*cos(theta)],
[                                                                                                            (w_y*sin(phi) + w_z*cos(phi))/cos(theta)],
[                                                                                                                         w_y*cos(phi) - w_z*sin(phi)],
[                                                                                             w_x + w_y*sin(phi)*tan(theta) + w_z*cos(phi)*tan(theta)],
[                                                                              

In [48]:
m = eoms.x_sym
n = eoms.u_sym

A = f_num.jacobian(m)
B = f_num.jacobian(n)

px, py, pz, phi, theta, psi, vx, vy, vz, p, q, r = m
tau1, tau2, tau3, tau4 = n

m_e = {
    px: 0.0,
    py: 0.0,
    pz: 0.0,
    phi: 0.0,
    theta: 0.0,
    psi: 0.0,
    vx: 0.0,
    vy: 0.0,
    vz: 0.0,
    p: 0.0,
    q: 0.0,
    r: 0.0
}

f_num_evaluated = f_num.subs({**m_e})
n_e = solve(f_num_evaluated, n)

A_num = np.array(A.subs({**m_e, **n_e}), dtype=float)
B_num = np.array(B.subs({**m_e, **n_e}), dtype=float)

In [49]:
Q = np.diag([10, 10, 10, 100, 100, 100, 1, 1, 1, 1, 1, 1])
R = np.diag([0.1, 0.1, 0.1, 0.1])

K = lqr(A_num, B_num, Q, R)